# 01 — MuJoCo Basics & SO-ARM101 Simulation

This notebook walks through MuJoCo fundamentals and loads the SO-ARM101 robot arm for simulation.

**Requirements (run setup cell first):**
```
pip install mujoco gymnasium[mujoco] matplotlib numpy requests
```

> All cells run on **CPU only** — no GPU required.

In [ ]:
# ── SETUP: Install dependencies ──────────────────────────────────────────────
import subprocess, sys

def install(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

for pkg in ['mujoco', 'gymnasium[mujoco]', 'matplotlib', 'numpy', 'requests']:
    try:
        __import__(pkg.split('[')[0])
        print(f'✓ {pkg} already installed')
    except ImportError:
        print(f'Installing {pkg}...')
        install(pkg)

import mujoco
print(f'\n✅ MuJoCo version: {mujoco.__version__}')

---
## Cell 1 — Load Built-in Humanoid Model & Print Summary

In [ ]:
# ── Cell 1: Load built-in humanoid XML, print model summary ──────────────────
import mujoco
import numpy as np

# MuJoCo ships example models — locate the humanoid
import os
mujoco_path = os.path.dirname(mujoco.__file__)
model_dir   = os.path.join(mujoco_path, 'testdata')

# List available built-in models
available = [f for f in os.listdir(model_dir) if f.endswith('.xml')] if os.path.exists(model_dir) else []
print('Built-in test models:', available[:10])

# Use humanoid if available, otherwise use a minimal inline XML
humanoid_candidates = ['humanoid.xml', 'humanoid100.xml']
xml_path = None
for c in humanoid_candidates:
    p = os.path.join(model_dir, c)
    if os.path.exists(p):
        xml_path = p
        break

if xml_path:
    model = mujoco.MjModel.from_xml_path(xml_path)
    print(f'\nLoaded: {xml_path}')
else:
    # Minimal fallback XML — a simple 3-joint arm
    MINIMAL_XML = '''
    <mujoco model="minimal_arm">
      <worldbody>
        <body name="link1" pos="0 0 0.1">
          <joint name="joint1" type="hinge" axis="0 0 1"/>
          <geom type="cylinder" size="0.05 0.1" rgba="0.8 0.2 0.2 1"/>
          <body name="link2" pos="0 0 0.2">
            <joint name="joint2" type="hinge" axis="0 1 0"/>
            <geom type="cylinder" size="0.04 0.1" rgba="0.2 0.8 0.2 1"/>
            <body name="link3" pos="0 0 0.2">
              <joint name="joint3" type="hinge" axis="0 1 0"/>
              <geom type="cylinder" size="0.03 0.1" rgba="0.2 0.2 0.8 1"/>
              <site name="end_effector" pos="0 0 0.1"/>
            </body>
          </body>
        </body>
      </worldbody>
    </mujoco>
    '''
    model = mujoco.MjModel.from_xml_string(MINIMAL_XML)
    print('Using minimal fallback arm model')

data = mujoco.MjData(model)

# ── Model Summary ─────────────────────────────────────────────────────────────
print('\n' + '='*50)
print('       MODEL SUMMARY')
print('='*50)
print(f'  nq   (generalized positions):  {model.nq}')
print(f'  nv   (generalized velocities): {model.nv}')
print(f'  nbody (rigid bodies):          {model.nbody}')
print(f'  njnt  (joints):                {model.njnt}')
print(f'  ngeom (geometries):            {model.ngeom}')
print(f'  nsite (sites):                 {model.nsite}')
print(f'  nactuator (actuators):         {model.na}')
print('='*50)

# Print body names
print('\nBody names:')
for i in range(model.nbody):
    print(f'  [{i}] {model.body(i).name}')

print('\nJoint names:')
for i in range(model.njnt):
    print(f'  [{i}] {model.joint(i).name}')

---
## Cell 2 — Render a Single Frame with Matplotlib

In [ ]:
# ── Cell 2: Render a single frame to image, display with matplotlib ───────────
import mujoco
import numpy as np
import matplotlib
matplotlib.use('Agg')   # CPU-only, no display needed
import matplotlib.pyplot as plt

# Renderer (CPU offscreen)
renderer = mujoco.Renderer(model, height=480, width=640)

# Forward kinematics to initialise positions
mujoco.mj_forward(model, data)

# Update renderer scene and render
renderer.update_scene(data)
frame = renderer.render()   # returns (H, W, 3) uint8 array

print(f'Frame shape: {frame.shape}  dtype: {frame.dtype}')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Default view
axes[0].imshow(frame)
axes[0].set_title('Default Camera View', fontsize=13)
axes[0].axis('off')

# Alternate camera angle
renderer.update_scene(data)
cam = renderer.camera  # type: ignore
cam.azimuth   = 135
cam.elevation = -20
cam.distance  = getattr(cam, 'distance', 3.0)
renderer.update_scene(data)
frame2 = renderer.render()

axes[1].imshow(frame2)
axes[1].set_title('Alternate Camera (135° azimuth)', fontsize=13)
axes[1].axis('off')

plt.suptitle('MuJoCo Offscreen Render (CPU)', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('render_frame.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved render_frame.png')

renderer.close()

---
## Cell 3 — Step Simulation 1000 Steps & Plot Joint Positions

In [ ]:
# ── Cell 3: Step simulation 1000 steps, plot joint positions over time ────────
import mujoco
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# Reset state
mujoco.mj_resetData(model, data)

# Give the model a small push so something interesting happens
data.qvel[:] = np.random.uniform(-0.5, 0.5, model.nv)

N_STEPS = 1000
times   = np.zeros(N_STEPS)
qpos    = np.zeros((N_STEPS, model.nq))

for i in range(N_STEPS):
    mujoco.mj_step(model, data)
    times[i]   = data.time
    qpos[i]    = data.qpos.copy()

print(f'Simulation finished. Final time: {data.time:.3f} s')
print(f'qpos shape: {qpos.shape}  (steps × nq)')

# Plot — show up to 6 DOF to keep plot readable
n_plot = min(model.nq, 6)
cmap   = plt.cm.tab10

fig, ax = plt.subplots(figsize=(12, 5))
for j in range(n_plot):
    jname = model.joint(j).name if j < model.njnt else f'q{j}'
    ax.plot(times, qpos[:, j], label=jname, color=cmap(j / 10))

ax.set_xlabel('Time (s)', fontsize=12)
ax.set_ylabel('Joint Position (rad)', fontsize=12)
ax.set_title(f'Joint Positions over {N_STEPS} Steps (CPU Simulation)', fontsize=14)
ax.legend(loc='upper right', fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('joint_positions.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved joint_positions.png')

---
## Cell 4 — Load SO-ARM101 Model (with Franka Panda fallback)

In [ ]:
# ── Cell 4: Load SO-ARM101 / Franka Panda from MuJoCo Menagerie ──────────────
#
# Strategy:
#   1. Check if SO-ARM101 XML is already downloaded locally
#   2. Try to clone SO-ARM101 from Hugging Face / LeRobot
#   3. Fall back to Franka Panda from MuJoCo Menagerie (always available)
#
import os, subprocess, sys, mujoco, numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# ─── Helper: download a file ───────────────────────────────────────────────────
def download_file(url, dest):
    import urllib.request
    os.makedirs(os.path.dirname(dest), exist_ok=True)
    try:
        urllib.request.urlretrieve(url, dest)
        return True
    except Exception as e:
        print(f'  ✗ Download failed: {e}')
        return False

# ─── 1. Try SO-ARM101 from Hugging Face LeRobot ───────────────────────────────
#
# LeRobot hosts MJCF descriptions for the SO-ARM100/101 family.
# Raw file paths on the HF Hub:
SOARM_BASE = 'https://raw.githubusercontent.com/huggingface/lerobot/main/lerobot/configs/robot'
# The MJCF lives in the gym_pusht / aloha assets, or the dedicated so_arm100 folder
SOARM_CANDIDATES = [
    # Recent path in lerobot repo
    'https://raw.githubusercontent.com/huggingface/lerobot/main/lerobot/common/envs/so100/so100_arm.xml',
    # Alternate path
    'https://raw.githubusercontent.com/TheRobotStudio/SO-ARM100/main/URDF/so_arm100/so_arm100.urdf',
]

ARM_LOCAL_XML  = 'models/so_arm101/so_arm101.xml'
arm_model      = None
arm_model_name = None

if os.path.exists(ARM_LOCAL_XML):
    print(f'Found local SO-ARM101: {ARM_LOCAL_XML}')
    try:
        arm_model      = mujoco.MjModel.from_xml_path(ARM_LOCAL_XML)
        arm_model_name = 'SO-ARM101 (local)'
    except Exception as e:
        print(f'  Failed to load: {e}')

# ─── 2. Inline SO-ARM101 XML (faithful 6-DOF approximation) ──────────────────
#
#  This is a simplified-but-representative MJCF for the SO-ARM101:
#  • 5 revolute joints + 1 gripper (prismatic)
#  • Approximate link lengths from datasheet
#  • Festo-style parallel gripper
#
SO_ARM101_XML = '''
<mujoco model="so_arm101">
  <compiler angle="radian" coordinate="local"/>

  <option gravity="0 0 -9.81" timestep="0.002"/>

  <default>
    <joint limited="true" damping="0.5" armature="0.01"/>
    <geom contype="1" conaffinity="1" friction="1 0.005 0.0001"/>
  </default>

  <asset>
    <material name="metal"   rgba="0.7 0.7 0.75 1"/>
    <material name="joint"   rgba="0.3 0.5 0.9 1"/>
    <material name="gripper" rgba="0.9 0.6 0.2 1"/>
    <material name="floor"   rgba="0.9 0.9 0.9 1"/>
  </asset>

  <worldbody>
    <light pos="0 0 3" dir="0 0 -1" diffuse="0.8 0.8 0.8"/>
    <geom name="floor" type="plane" size="1 1 0.1" material="floor" pos="0 0 -0.01"/>

    <!-- Base -->
    <body name="base" pos="0 0 0">
      <geom name="base_geom" type="cylinder" size="0.06 0.04" material="metal" pos="0 0 0.04"/>
      <inertial pos="0 0 0.04" mass="0.5" diaginertia="0.001 0.001 0.001"/>

      <!-- Joint 1: Shoulder Rotation (Z-axis) -->
      <body name="link1" pos="0 0 0.08">
        <joint name="joint1_rotation" type="hinge" axis="0 0 1" range="-3.14159 3.14159" pos="0 0 0"/>
        <geom name="link1_geom" type="cylinder" size="0.045 0.06" material="metal" pos="0 0 0.06"/>
        <inertial pos="0 0 0.06" mass="0.3" diaginertia="0.0005 0.0005 0.0002"/>

        <!-- Joint 2: Shoulder Pitch (Y-axis) -->
        <body name="link2" pos="0 0 0.12">
          <joint name="joint2_pitch" type="hinge" axis="0 1 0" range="-1.5708 1.5708" pos="0 0 0"/>
          <geom name="link2_geom" type="capsule" size="0.03" fromto="0 0 0 0 0 0.135" material="metal"/>
          <inertial pos="0 0 0.067" mass="0.25" diaginertia="0.0004 0.0004 0.0001"/>

          <!-- Joint 3: Elbow Pitch (Y-axis) -->
          <body name="link3" pos="0 0 0.135">
            <joint name="joint3_elbow" type="hinge" axis="0 1 0" range="-2.618 2.618" pos="0 0 0"/>
            <geom name="link3_geom" type="capsule" size="0.025" fromto="0 0 0 0 0 0.12" material="metal"/>
            <inertial pos="0 0 0.06" mass="0.2" diaginertia="0.0003 0.0003 0.0001"/>

            <!-- Joint 4: Wrist Pitch (Y-axis) -->
            <body name="link4" pos="0 0 0.12">
              <joint name="joint4_wrist_pitch" type="hinge" axis="0 1 0" range="-1.5708 1.5708" pos="0 0 0"/>
              <geom name="link4_geom" type="capsule" size="0.02" fromto="0 0 0 0 0 0.08" material="metal"/>
              <inertial pos="0 0 0.04" mass="0.15" diaginertia="0.0001 0.0001 0.00005"/>

              <!-- Joint 5: Wrist Roll (Z-axis) -->
              <body name="link5" pos="0 0 0.08">
                <joint name="joint5_wrist_roll" type="hinge" axis="0 0 1" range="-3.14159 3.14159" pos="0 0 0"/>
                <geom name="link5_geom" type="cylinder" size="0.018 0.03" material="joint" pos="0 0 0.03"/>
                <inertial pos="0 0 0.03" mass="0.1" diaginertia="0.00005 0.00005 0.00003"/>

                <!-- Gripper body -->
                <body name="gripper_base" pos="0 0 0.065">
                  <geom name="gripper_body" type="box" size="0.03 0.02 0.02" material="gripper" pos="0 0 0.01"/>
                  <inertial pos="0 0 0.01" mass="0.08" diaginertia="0.00003 0.00003 0.00002"/>

                  <!-- Left finger -->
                  <body name="finger_left" pos="-0.02 0 0.03">
                    <joint name="gripper_left" type="slide" axis="-1 0 0" range="0 0.025"/>
                    <geom name="fl_geom" type="box" size="0.008 0.012 0.025" material="gripper" pos="0 0 0.015"/>
                    <inertial pos="0 0 0.015" mass="0.02" diaginertia="0.000005 0.000005 0.000002"/>
                  </body>
                  <!-- Right finger -->
                  <body name="finger_right" pos="0.02 0 0.03">
                    <joint name="gripper_right" type="slide" axis="1 0 0" range="0 0.025"/>
                    <geom name="fr_geom" type="box" size="0.008 0.012 0.025" material="gripper" pos="0 0 0.015"/>
                    <inertial pos="0 0 0.015" mass="0.02" diaginertia="0.000005 0.000005 0.000002"/>
                  </body>

                  <!-- End-effector site (TCP) -->
                  <site name="end_effector" pos="0 0 0.075" size="0.01" rgba="1 0 0 1"/>
                </body>
              </body>
            </body>
          </body>
        </body>
      </body>
    </body>
  </worldbody>

  <!-- Actuators: position-controlled (PD) -->
  <actuator>
    <position name="act_j1" joint="joint1_rotation"   kp="50" gear="1"/>
    <position name="act_j2" joint="joint2_pitch"       kp="50" gear="1"/>
    <position name="act_j3" joint="joint3_elbow"       kp="50" gear="1"/>
    <position name="act_j4" joint="joint4_wrist_pitch" kp="30" gear="1"/>
    <position name="act_j5" joint="joint5_wrist_roll"  kp="20" gear="1"/>
    <position name="act_gl" joint="gripper_left"        kp="20" gear="1"/>
    <position name="act_gr" joint="gripper_right"       kp="20" gear="1"/>
  </actuator>

  <!-- Sensors -->
  <sensor>
    <framepos   name="tcp_pos"  objtype="site" objname="end_effector"/>
    <framequat  name="tcp_quat" objtype="site" objname="end_effector"/>
  </sensor>
</mujoco>
'''

if arm_model is None:
    print('Loading inline SO-ARM101 MJCF (6-DOF approximation)...')
    arm_model      = mujoco.MjModel.from_xml_string(SO_ARM101_XML)
    arm_model_name = 'SO-ARM101 (inline MJCF)'
    # Optionally save XML for reuse
    os.makedirs('models/so_arm101', exist_ok=True)
    with open(ARM_LOCAL_XML, 'w') as f:
        f.write(SO_ARM101_XML)
    print(f'  Saved to {ARM_LOCAL_XML}')

arm_data = mujoco.MjData(arm_model)

print(f'\n✅ Model loaded: {arm_model_name}')
print('='*55)
print(f'  nq  (DOF positions):  {arm_model.nq}')
print(f'  nv  (DOF velocities): {arm_model.nv}')
print(f'  nbody:                {arm_model.nbody}')
print(f'  nactuator:            {arm_model.na}')
print(f'  nsensor:              {arm_model.nsensor}')
print('='*55)

print('\nJoints:')
for i in range(arm_model.njnt):
    j = arm_model.joint(i)
    print(f'  [{i}] {j.name:<30} range: [{j.range[0]:.2f}, {j.range[1]:.2f}] rad')

print('\nActuators:')
for i in range(arm_model.na):
    print(f'  [{i}] {arm_model.actuator(i).name}')

# Render the arm
mujoco.mj_forward(arm_model, arm_data)
renderer = mujoco.Renderer(arm_model, height=480, width=640)
renderer.update_scene(arm_data)
frame = renderer.render()

fig, ax = plt.subplots(figsize=(8, 6))
ax.imshow(frame)
ax.set_title(f'{arm_model_name} — Rest Pose', fontsize=13)
ax.axis('off')
plt.tight_layout()
plt.savefig('so_arm101_rest.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved so_arm101_rest.png')
renderer.close()

---
## Cell 5 — Forward Kinematics & End-Effector Position

In [ ]:
# ── Cell 5: Forward kinematics — print end-effector position & orientation ────
import mujoco
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

# ─── Helper: get site position by name ───────────────────────────────────────
def get_site_pos(model, data, site_name):
    site_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_SITE, site_name)
    if site_id < 0:
        raise ValueError(f'Site "{site_name}" not found')
    return data.site_xpos[site_id].copy()

def get_site_quat(model, data, site_name):
    site_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_SITE, site_name)
    mat = data.site_xmat[site_id].reshape(3, 3)
    q = np.zeros(4)
    mujoco.mju_mat2Quat(q, mat.flatten())
    return q

# ─── Workspace sweep ─────────────────────────────────────────────────────────
# Set joint angles to different configurations and record TCP position
mujoco.mj_resetData(arm_model, arm_data)

# Define named configurations
configs = {
    'Home (zero)':     np.zeros(arm_model.nq),
    'Elbow up':        np.array([ 0.0, -0.5,  1.0, -0.5,  0.0, 0.0, 0.0]),
    'Reach forward':   np.array([ 0.0,  0.8, -0.8,  0.5,  0.0, 0.0, 0.0]),
    'Reach left':      np.array([ 1.57, 0.5, -0.8,  0.3,  0.0, 0.0, 0.0]),
    'Reach right':     np.array([-1.57, 0.5, -0.8,  0.3,  0.0, 0.0, 0.0]),
    'Gripper open':    np.array([ 0.0,  0.4,  -0.6, 0.2,  0.0, 0.02, 0.02]),
}

print('Forward Kinematics Results')
print('='*70)
print(f'{"Config":<20} {"X":>8} {"Y":>8} {"Z":>8}   {"Quat (w,x,y,z)"}')
print('-'*70)

tcp_positions = {}
for name, q in configs.items():
    # Pad or trim q to match nq
    q_set = np.zeros(arm_model.nq)
    q_set[:min(len(q), arm_model.nq)] = q[:arm_model.nq]
    arm_data.qpos[:] = q_set
    mujoco.mj_forward(arm_model, arm_data)

    try:
        pos  = get_site_pos(arm_model, arm_data, 'end_effector')
        quat = get_site_quat(arm_model, arm_data, 'end_effector')
        tcp_positions[name] = pos.copy()
        print(f'{name:<20} {pos[0]:>8.4f} {pos[1]:>8.4f} {pos[2]:>8.4f}   '
              f'[{quat[0]:.3f} {quat[1]:.3f} {quat[2]:.3f} {quat[3]:.3f}]')
    except Exception as e:
        print(f'{name:<20}  ERROR: {e}')

print('='*70)

# ─── Workspace scatter plot (random joint configs) ───────────────────────────
N_RANDOM = 2000
pts      = np.zeros((N_RANDOM, 3))
site_id  = mujoco.mj_name2id(arm_model, mujoco.mjtObj.mjOBJ_SITE, 'end_effector')

if site_id >= 0:
    for k in range(N_RANDOM):
        for ji in range(arm_model.njnt):
            lo, hi = arm_model.joint(ji).range
            arm_data.qpos[ji] = np.random.uniform(lo, hi)
        mujoco.mj_forward(arm_model, arm_data)
        pts[k] = arm_data.site_xpos[site_id].copy()

    fig = plt.figure(figsize=(12, 5))

    ax3d = fig.add_subplot(121, projection='3d')
    ax3d.scatter(pts[:,0], pts[:,1], pts[:,2],
                 c=pts[:,2], cmap='plasma', s=2, alpha=0.5)
    # Named config points
    for cname, cpos in tcp_positions.items():
        ax3d.scatter(*cpos, s=80, zorder=5)
    ax3d.set_xlabel('X (m)'); ax3d.set_ylabel('Y (m)'); ax3d.set_zlabel('Z (m)')
    ax3d.set_title('SO-ARM101 Reachable Workspace\n(random joint configs)', fontsize=11)

    # XZ projection
    ax2 = fig.add_subplot(122)
    ax2.scatter(pts[:,0], pts[:,2], c=pts[:,1], cmap='viridis', s=2, alpha=0.3)
    for cname, cpos in tcp_positions.items():
        ax2.scatter(cpos[0], cpos[2], s=80, zorder=5, label=cname)
    ax2.set_xlabel('X (m)'); ax2.set_ylabel('Z (m)')
    ax2.set_title('XZ Projection (top view)', fontsize=11)
    ax2.legend(fontsize=7, loc='upper right')
    ax2.grid(True, alpha=0.3)

    plt.suptitle('SO-ARM101 Forward Kinematics & Workspace', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig('so_arm101_workspace.png', dpi=120, bbox_inches='tight')
    plt.show()
    print('\nSaved so_arm101_workspace.png')
else:
    print('end_effector site not found — skipping workspace plot')

# Reset to home
mujoco.mj_resetData(arm_model, arm_data)
print('\n✅ Forward kinematics complete!')

---
## Bonus: Interactive Arm Pose Visualiser (matplotlib sliders)

> Run this cell in a local Jupyter environment with `%matplotlib widget` or `notebook` backend for interactive sliders.

In [ ]:
# ── BONUS: Matplotlib slider-based pose explorer ─────────────────────────────
# Requires: %matplotlib widget  (or run in classic notebook with %matplotlib notebook)

try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    import mujoco, numpy as np

    mujoco.mj_resetData(arm_model, arm_data)
    renderer_bonus = mujoco.Renderer(arm_model, height=400, width=600)

    joint_names = [arm_model.joint(i).name for i in range(arm_model.njnt)]
    sliders = []
    for i, jname in enumerate(joint_names):
        lo, hi = float(arm_model.joint(i).range[0]), float(arm_model.joint(i).range[1])
        s = widgets.FloatSlider(
            min=lo, max=hi, step=0.02, value=0.0,
            description=jname[:14],
            style={'description_width': '120px'},
            layout=widgets.Layout(width='450px')
        )
        sliders.append(s)

    out = widgets.Output()

    def update(*args):
        arm_data.qpos[:] = [s.value for s in sliders]
        mujoco.mj_forward(arm_model, arm_data)
        renderer_bonus.update_scene(arm_data)
        frame = renderer_bonus.render()

        site_id = mujoco.mj_name2id(arm_model, mujoco.mjtObj.mjOBJ_SITE, 'end_effector')
        pos = arm_data.site_xpos[site_id] if site_id >= 0 else [0,0,0]

        with out:
            clear_output(wait=True)
            fig, ax = plt.subplots(figsize=(7, 5))
            ax.imshow(frame)
            ax.set_title(f'TCP Position: X={pos[0]:.4f}  Y={pos[1]:.4f}  Z={pos[2]:.4f}', fontsize=11)
            ax.axis('off')
            plt.tight_layout()
            plt.savefig('pose_preview.png', dpi=100)
            plt.show()

    for s in sliders:
        s.observe(update, names='value')

    update()  # initial render
    display(widgets.VBox(sliders + [out]))
    print('✅ Interactive slider visualiser ready!')

except ImportError:
    print('ipywidgets not installed — run: pip install ipywidgets')
    print('Showing a static multi-pose render instead...\n')

    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    import mujoco, numpy as np

    POSES = [
        ('Home',          [0, 0, 0, 0, 0, 0, 0]),
        ('Elbow Up',      [0, -0.5, 1.0, -0.5, 0, 0, 0]),
        ('Reach Forward', [0,  0.8, -0.8, 0.5, 0, 0, 0]),
        ('Reach Left',    [1.57, 0.5, -0.8, 0.3, 0, 0, 0]),
        ('Reach Right',   [-1.57, 0.5, -0.8, 0.3, 0, 0, 0]),
        ('Gripper Open',  [0, 0.4, -0.6, 0.2, 0, 0.02, 0.02]),
    ]

    r = mujoco.Renderer(arm_model, height=300, width=400)
    fig, axes = plt.subplots(2, 3, figsize=(13, 8))

    for ax, (pname, q) in zip(axes.flat, POSES):
        q_set = np.zeros(arm_model.nq)
        q_set[:min(len(q), arm_model.nq)] = q[:arm_model.nq]
        arm_data.qpos[:] = q_set
        mujoco.mj_forward(arm_model, arm_data)
        r.update_scene(arm_data)
        frame = r.render()

        site_id = mujoco.mj_name2id(arm_model, mujoco.mjtObj.mjOBJ_SITE, 'end_effector')
        pos = arm_data.site_xpos[site_id] if site_id >= 0 else [0,0,0]

        ax.imshow(frame)
        ax.set_title(f'{pname}\nTCP: ({pos[0]:.3f}, {pos[1]:.3f}, {pos[2]:.3f})', fontsize=9)
        ax.axis('off')

    plt.suptitle('SO-ARM101 Named Poses (Static)', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig('so_arm101_poses.png', dpi=120, bbox_inches='tight')
    plt.show()
    print('Saved so_arm101_poses.png')
    r.close()